In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA version: 12.1
Device count: 1
GPU name: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
import pandas as pd
import os
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
# Display all columns
pd.set_option('display.max_columns', None)

# Display all rows
pd.set_option('display.max_rows', None)

# Set the width to show all content in each cell
pd.set_option('display.width', None)

# Set the max string length to display
pd.set_option('display.max_colwidth', None)

In [5]:
bbox = [
    28.0,   # north
    85.5,   # west
    20.0,   # south
    90.5    # east
]

#### GEE raw data handling

In [5]:
import sys
sys.path.append('C:/Users/rishe/Dissertation')

In [32]:
import pandas as pd
import glob

files = sorted(glob.glob("C:/Users/rishe/Dissertation/data/gee_era5_data/ERA5_Station_Data_*.csv"))

df_list = []

for f in files:
    print(f"Loading {f}")
    df = pd.read_csv(f)
    df_list.append(df)

df = pd.concat(df_list, ignore_index=True)

print(df.shape)

Loading C:/Users/rishe/Dissertation/data/gee_era5_data\ERA5_Station_Data_1970.csv
Loading C:/Users/rishe/Dissertation/data/gee_era5_data\ERA5_Station_Data_1971.csv
Loading C:/Users/rishe/Dissertation/data/gee_era5_data\ERA5_Station_Data_1972.csv
Loading C:/Users/rishe/Dissertation/data/gee_era5_data\ERA5_Station_Data_1973.csv
Loading C:/Users/rishe/Dissertation/data/gee_era5_data\ERA5_Station_Data_1974.csv
Loading C:/Users/rishe/Dissertation/data/gee_era5_data\ERA5_Station_Data_1975.csv
Loading C:/Users/rishe/Dissertation/data/gee_era5_data\ERA5_Station_Data_1976.csv
Loading C:/Users/rishe/Dissertation/data/gee_era5_data\ERA5_Station_Data_1977.csv
Loading C:/Users/rishe/Dissertation/data/gee_era5_data\ERA5_Station_Data_1978.csv
Loading C:/Users/rishe/Dissertation/data/gee_era5_data\ERA5_Station_Data_1979.csv
Loading C:/Users/rishe/Dissertation/data/gee_era5_data\ERA5_Station_Data_1980.csv
Loading C:/Users/rishe/Dissertation/data/gee_era5_data\ERA5_Station_Data_1981.csv
Loading C:/Users

In [33]:
print(df.columns)

Index(['date', 'station_id', 'total_precipitation_sum', 'temperature_2m',
       'dewpoint_temperature_2m', 'surface_pressure',
       'u_component_of_wind_10m', 'v_component_of_wind_10m'],
      dtype='object')


In [34]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['station_id', 'date'])

In [35]:
# Check for dropped coastal/edge stations
original_stations = pd.read_csv('C:/Users/rishe/Dissertation/data/wb_station_coords.csv')['station_id'].unique()
extracted_stations = df['station_id'].unique()
missing_stations = set(original_stations) - set(extracted_stations)

if missing_stations:
    print(f"⚠️ Warning: The following stations fell outside the ERA5-Land mask: {missing_stations}")

⚠️ Warning: The following stations fell outside the ERA5-Land mask: {'SAGAR ISLAND', 'SANDHEADS'}


#### Unit conversion

In [36]:
# Rain: meters → mm
df['rain'] = df['total_precipitation_sum'] * 1000

# Temperature: Kelvin → Celsius
df['temp'] = df['temperature_2m'] - 273.15
df['dew'] = df['dewpoint_temperature_2m'] - 273.15

# Pressure: Pa → hPa
df['pressure'] = df['surface_pressure'] / 100

# Wind stays same
df['u10'] = df['u_component_of_wind_10m']
df['v10'] = df['v_component_of_wind_10m']

In [37]:
df = df[['date', 'station_id', 'rain', 'temp', 'dew', 'pressure', 'u10', 'v10']]

#### pivoting

In [39]:
rain_pivot = df.pivot(index='date', columns='station_id', values='rain')

In [8]:
temp_pivot = df.pivot(index='date', columns='station_id', values='temp')
dew_pivot = df.pivot(index='date', columns='station_id', values='dew')
pressure_pivot = df.pivot(index='date', columns='station_id', values='pressure')
u10_pivot = df.pivot(index='date', columns='station_id', values='u10')
v10_pivot = df.pivot(index='date', columns='station_id', values='v10')
print("Rain pivot shape:", rain_pivot.shape)
print("Temperature pivot shape:", temp_pivot.shape) 
print("Dew point pivot shape:", dew_pivot.shape)
print("Pressure pivot shape:", pressure_pivot.shape)
print("U10 pivot shape:", u10_pivot.shape)
print("V10 pivot shape:", v10_pivot.shape)

Rain pivot shape: (19723, 291)
Temperature pivot shape: (19723, 291)
Dew point pivot shape: (19723, 291)
Pressure pivot shape: (19723, 291)
U10 pivot shape: (19723, 291)
V10 pivot shape: (19723, 291)


In [11]:
# Check current date range
print("Current date ranges:")
print(f"rain_pivot: {rain_pivot.index.min()} to {rain_pivot.index.max()}")
print(f"temp_pivot: {temp_pivot.index.min()} to {temp_pivot.index.max()}")
print(f"dew_pivot: {dew_pivot.index.min()} to {dew_pivot.index.max()}")

# Check for missing values in each dataframe
print("\nMissing values (counts):")
print(f"rain_pivot: {rain_pivot.isna().sum().sum()} / {rain_pivot.size}")
print(f"temp_pivot: {temp_pivot.isna().sum().sum()} / {temp_pivot.size}")
print(f"dew_pivot: {dew_pivot.isna().sum().sum()} / {dew_pivot.size}")

# Check missing values per column
print("\nMissing values per station (rain_pivot):")
print(rain_pivot.isna().sum())

# Check for gaps in date index
print("\nDate gaps (days missing from index):")
full_range = pd.date_range(start=rain_pivot.index.min(), end=rain_pivot.index.max(), freq='D')
missing_dates = full_range.difference(rain_pivot.index)
print(f"Total missing dates: {len(missing_dates)}")
if len(missing_dates) > 0:
    print(f"Example missing dates: {missing_dates[:10]}")

# Check sparsity
print(f"\nExpected days: {len(full_range)}")
print(f"Actual days in index: {len(rain_pivot)}")
print(f"Sparsity: {(1 - len(rain_pivot)/len(full_range))*100:.2f}%")

Current date ranges:
rain_pivot: 1970-01-01 00:00:00 to 2023-12-31 00:00:00
temp_pivot: 1970-01-01 00:00:00 to 2023-12-31 00:00:00
dew_pivot: 1970-01-01 00:00:00 to 2023-12-31 00:00:00

Missing values (counts):
rain_pivot: 0 / 5739393
temp_pivot: 0 / 5739393
dew_pivot: 0 / 5739393

Missing values per station (rain_pivot):
station_id
AKRIGANJ           0
ALGARAH            0
ALIPUR             0
ALIPURDUAR         0
ALIPURDUAR(CWC)    0
                  ..
TOOFANGANJ         0
TUFANGANJ          0
TUSUMA(CWC)        0
ULUBERIA           0
VISHNUPUR          0
Length: 291, dtype: int64

Date gaps (days missing from index):
Total missing dates: 0

Expected days: 19723
Actual days in index: 19723
Sparsity: 0.00%


In [12]:
# After Step 5, force a complete date index
full_date_range = pd.date_range(start='1970-01-01', end='2023-12-31', freq='D')

rain_pivot = rain_pivot.reindex(full_date_range)
temp_pivot = temp_pivot.reindex(full_date_range)
dew_pivot = dew_pivot.reindex(full_date_range)
pressure_pivot = pressure_pivot.reindex(full_date_range)
u10_pivot = u10_pivot.reindex(full_date_range)
v10_pivot = v10_pivot.reindex(full_date_range)

# Now forward-fill or interpolate any missing days
rain_pivot = rain_pivot.interpolate(method='time')
temp_pivot = temp_pivot.interpolate(method='time')
dew_pivot = dew_pivot.interpolate(method='time')
pressure_pivot = pressure_pivot.interpolate(method='time')
u10_pivot = u10_pivot.interpolate(method='time')
v10_pivot = v10_pivot.interpolate(method='time')

#### Comparison with existing data

In [13]:
OLD_DATA_PATH = f'C:/Users/rishe/Dissertation/data/processed_rain_with_coords_final.parquet'
df_old = pd.read_parquet(OLD_DATA_PATH)

In [14]:
stations = df_old['station_id'].unique()

# station_to_idx = {s:i for i,s in enumerate(stations)}
# idx_to_station = {i:s for s,i in station_to_idx.items()}

# df['node_id'] = df['station_id'].map(station_to_idx)

# num_nodes = len(stations)

# print("Total stations:", num_nodes)

In [15]:
pivot = df_old.pivot(index='date',
                 columns='station_id',
                 values='rainfall')

pivot = pivot[stations]

In [16]:
pivot.head()

station_id,AKRIGANJ,ALGARAH,ALIPUR,ALIPURDUAR,ALIPURDUAR(CWC),ALSHA,AMBIKANAGAR,AMLAGORA,AMTA,AMTALA,...,TAPAN,TARAKESHWAR,TEESTA BAZAR,TENTULIA,TILPARA BARRAGE,TOOFANGANJ,TUFANGANJ,TUSUMA(CWC),ULUBERIA,VISHNUPUR
date,,,,,,,,,,,,,,,,,,,,,
1901-01-01,4.1,NaN,0.0,2.8,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,0.0,0.0
1901-01-02,3.3,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2.5,NaN,0.0,11.7
1901-01-03,0.3,NaN,0.5,0.0,NaN,NaN,NaN,8.9,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,1.8,7.6
1901-01-04,0.3,NaN,12.7,0.0,NaN,NaN,NaN,4.6,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,15.2,10.2
1901-01-05,0.0,NaN,13.0,0.0,NaN,NaN,NaN,2.8,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,12.7,7.4


In [17]:
common_dates = rain_pivot.index.intersection(pivot.index)

rain_pivot = rain_pivot.loc[common_dates]
pivot = pivot.loc[common_dates]

In [19]:
stations_to_remove = {'SAGAR ISLAND', 'SANDHEADS'}

In [20]:
pivot = pivot.drop(columns=stations_to_remove, errors='ignore')
pivot.columns

Index(['AKRIGANJ', 'ALGARAH', 'ALIPUR', 'ALIPURDUAR', 'ALIPURDUAR(CWC)',
       'ALSHA', 'AMBIKANAGAR', 'AMLAGORA', 'AMTA', 'AMTALA',
       ...
       'TAPAN', 'TARAKESHWAR', 'TEESTA BAZAR', 'TENTULIA', 'TILPARA BARRAGE',
       'TOOFANGANJ', 'TUFANGANJ', 'TUSUMA(CWC)', 'ULUBERIA', 'VISHNUPUR'],
      dtype='object', name='station_id', length=291)

In [21]:
from scipy.stats import pearsonr

corrs = {}

for st in pivot.columns:
    obs = pivot[st]
    era = rain_pivot[st]

    mask = (~obs.isna()) & (~era.isna())

    if mask.sum() > 10:
        corrs[st] = pearsonr(obs[mask], era[mask])[0]

pd.Series(corrs).describe()

C:\Users\rishe\AppData\Local\Temp\ipykernel_16088\2029039052.py:12: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corrs[st] = pearsonr(obs[mask], era[mask])[0]


count    219.000000
mean       0.350846
std        0.093147
min       -0.095562
25%        0.320478
50%        0.364674
75%        0.399746
max        0.568759
dtype: float64

#### sanity checks

In [24]:
rain_pivot.isna().mean().sort_values()

station_id
AKRIGANJ            0.0
MO KOLKATA          0.0
MIDNAPORE(CWC)      0.0
MIDNAPORE (SCRF)    0.0
MIDNAPORE           0.0
                   ... 
GHATAL              0.0
GHARGRAM KHEJURI    0.0
GHARGRAM            0.0
DEBAGRAM            0.0
VISHNUPUR           0.0
Length: 291, dtype: float64

In [25]:
rain_pivot.index.to_series().diff().value_counts()

1 days    18992
Name: count, dtype: int64

In [26]:
rain_pivot.describe()

station_id,AKRIGANJ,ALGARAH,ALIPUR,ALIPURDUAR,ALIPURDUAR(CWC),ALSHA,AMBIKANAGAR,AMLAGORA,AMTA,AMTALA,...,TAPAN,TARAKESHWAR,TEESTA BAZAR,TENTULIA,TILPARA BARRAGE,TOOFANGANJ,TUFANGANJ,TUSUMA(CWC),ULUBERIA,VISHNUPUR
count,18993.000000,18993.000000,18993.000000,18993.000000,18993.000000,18993.000000,18993.000000,18993.000000,18993.000000,18993.000000,...,18993.000000,18993.000000,18993.000000,18993.000000,18993.000000,18993.000000,18993.000000,18993.000000,18993.000000,18993.000000
mean,4.287167,9.070627,4.409302,9.474326,9.474326,3.691983,3.646190,3.809662,4.066068,4.274747,...,4.988789,4.092470,8.899249,3.986270,3.813608,10.015051,10.015051,3.691466,4.234098,3.772588
std,8.492091,19.542090,8.997975,20.803200,20.803200,8.340680,8.482389,8.367021,8.479014,8.481363,...,10.059130,8.276741,18.455027,8.172036,7.769142,18.616052,18.616052,8.510710,8.655219,8.246267
min,-0.000030,-0.000022,-0.000030,-0.000030,-0.000030,-0.000030,-0.000030,-0.000030,-0.000030,-0.000030,...,-0.000030,-0.000030,-0.000030,-0.000030,-0.000030,-0.000030,-0.000030,-0.000030,-0.000030,-0.000030
25%,0.001287,0.367248,0.001520,0.002171,0.002171,0.001261,0.001281,0.001285,0.001295,0.001290,...,0.001293,0.001293,0.654858,0.001290,0.001281,0.002719,0.002719,0.001276,0.001299,0.001284
50%,0.344366,2.897283,0.341779,0.851059,0.851059,0.159764,0.191140,0.239184,0.260535,0.404430,...,0.534296,0.307622,3.528263,0.289166,0.262729,1.382570,1.382570,0.159442,0.326785,0.209481
75%,5.167335,9.206194,5.444903,10.743839,10.743839,3.943473,3.902440,4.305321,4.790574,5.135092,...,6.222451,4.952420,9.055614,4.743612,4.483187,13.398299,13.398299,3.836602,5.247727,4.228827
max,145.912248,463.592333,179.967478,415.576632,415.576632,161.574262,213.494822,205.702245,170.466349,152.282405,...,316.803545,151.921436,360.217690,142.185166,133.772349,247.232926,247.232926,184.250066,132.804796,301.946700


##### all pivots should have same index and columnn

In [40]:
assert rain_pivot.index.equals(temp_pivot.index)
assert rain_pivot.index.equals(dew_pivot.index)
assert rain_pivot.index.equals(pressure_pivot.index)
assert rain_pivot.index.equals(u10_pivot.index)
assert rain_pivot.index.equals(v10_pivot.index)

assert rain_pivot.columns.equals(temp_pivot.columns)

In [28]:
# Check index lengths
print("Index lengths:")
print(f"rain_pivot: {len(rain_pivot.index)}")
print(f"temp_pivot: {len(temp_pivot.index)}")
print(f"dew_pivot: {len(dew_pivot.index)}")

# Check if indices are equal
print("\nIndex equality checks:")
print(f"rain == temp: {rain_pivot.index.equals(temp_pivot.index)}")
print(f"rain == dew: {rain_pivot.index.equals(dew_pivot.index)}")
print(f"temp == dew: {temp_pivot.index.equals(dew_pivot.index)}")

# Check date ranges
print("\nDate ranges:")
print(f"rain_pivot: {rain_pivot.index.min()} to {rain_pivot.index.max()}")
print(f"temp_pivot: {temp_pivot.index.min()} to {temp_pivot.index.max()}")
print(f"dew_pivot: {dew_pivot.index.min()} to {dew_pivot.index.max()}")

# Check for missing dates in each
print("\nMissing dates count:")
full_range = pd.date_range(start='1970-01-01', end='2023-12-31', freq='D')
for name, df in [('rain', rain_pivot), ('temp', temp_pivot), ('dew', dew_pivot)]:
    missing = len(full_range) - len(df.index)
    print(f"{name}_pivot: {missing} missing dates")

Index lengths:
rain_pivot: 18993
temp_pivot: 19723
dew_pivot: 19723

Index equality checks:
rain == temp: False
rain == dew: False
temp == dew: True

Date ranges:
rain_pivot: 1970-01-01 00:00:00 to 2021-12-31 00:00:00
temp_pivot: 1970-01-01 00:00:00 to 2023-12-31 00:00:00
dew_pivot: 1970-01-01 00:00:00 to 2023-12-31 00:00:00

Missing dates count:
rain_pivot: 730 missing dates
temp_pivot: 0 missing dates
dew_pivot: 0 missing dates


In [42]:
df.tail()

,date,station_id,rain,temp,dew,pressure,u10,v10
5738228,2023-12-27,VISHNUPUR,0.000852,19.743502,14.070748,1009.799840,0.556254,-1.711068
5738519,2023-12-28,VISHNUPUR,0.000000,19.524543,14.400278,1009.358599,0.385949,-1.728519
5738810,2023-12-29,VISHNUPUR,0.000852,19.423895,13.560482,1008.917546,0.885672,-1.940993
5739101,2023-12-30,VISHNUPUR,0.000000,18.898136,12.990707,1008.293672,0.972421,-1.962241
5739392,2023-12-31,VISHNUPUR,0.000858,18.471246,13.216286,1008.341976,0.443991,-2.144200


In [43]:
save_path = f'C:/Users/rishe/Dissertation/data/era5_clean.parquet'
df.to_parquet(save_path, index=False)

In [45]:
pivot_save_path = f'C:/Users/rishe/Dissertation/data/era5_pivot_data/'

In [46]:
rain_pivot.to_parquet(f"{pivot_save_path}rain_pivot.parquet")
temp_pivot.to_parquet(f"{pivot_save_path}temp_pivot.parquet")
dew_pivot.to_parquet(f"{pivot_save_path}dew_pivot.parquet")
pressure_pivot.to_parquet(f"{pivot_save_path}pressure_pivot.parquet")
u10_pivot.to_parquet(f"{pivot_save_path}u10_pivot.parquet")
v10_pivot.to_parquet(f"{pivot_save_path}v10_pivot.parquet")


#### data scaling

In [ ]:
import numpy as np
import pandas as pd

# 1. Define your training cutoff to prevent data leakage
# Adjust this year based on how you plan to split your train/val/test
TRAIN_END_DATE = '2015-12-31' 

# Group your pivots in a dictionary for clean looping
pivots = {
    'rain': rain_pivot,
    'temp': temp_pivot,
    'dew': dew_pivot,
    'pressure': pressure_pivot,
    'u10': u10_pivot,
    'v10': v10_pivot
}

scaled_pivots = {}
scaler_params = {} # You WILL need these later to un-scale your predictions!

print("⚙️ Scaling features globally...")

for feature_name, df in pivots.items():
    # 2. Isolate the training data
    train_data = df.loc[:TRAIN_END_DATE].values
    
    # 3. Calculate global mean and std for this feature across ALL stations
    # We use np.nanmean just in case there are lingering NaNs
    mu = np.nanmean(train_data)
    sigma = np.nanstd(train_data)
    
    # Prevent division by zero (unlikely for weather data, but good practice)
    if sigma == 0:
        sigma = 1e-8
        
    # 4. Apply scaling to the ENTIRE dataset (Train + Val + Test)
    scaled_df = (df - mu) / sigma
    
    # Store the scaled dataframe and the parameters
    scaled_pivots[feature_name] = scaled_df
    scaler_params[feature_name] = {'mean': mu, 'std': sigma}
    
    print(f"✅ {feature_name: <8} | Mean: {mu: >8.2f} | Std: {sigma: >8.2f}")

# Extract them back out if you prefer working with individual variables
rain_pivot_scaled = scaled_pivots['rain']
temp_pivot_scaled = scaled_pivots['temp']
dew_pivot_scaled = scaled_pivots['dew']
pressure_pivot_scaled = scaled_pivots['pressure']
u10_pivot_scaled = scaled_pivots['u10']
v10_pivot_scaled = scaled_pivots['v10']